In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.3 Causal masking

> 🎯 **Goal:** Stop a token from attending to tokens that come after it, so the model only ever uses the past to predict the future.

**Why this matters.** A GPT-style decoder is trained to predict the next token. If, while predicting token 5, the model were allowed to look at tokens 6, 7, and 8, it would be cheating: the answer is right there. Causal masking enforces the rule that makes next-token training honest, and it is the single difference between a generic attention block and a *decoder* attention block.

**The intuition.** No peeking at the future. Imagine reading a sentence one word at a time with a card covering everything to the right of where you are. Each word can see itself and everything before it, but nothing after. The causal mask is that card.

**The mechanics.** The score matrix has a row per query token and a column per key token. "Token i attends to token j in the future" is exactly the upper triangle of that matrix (where `j > i`). To block it we set those upper-triangle scores to negative infinity *before* the softmax. Why `-inf` and not 0? Because `softmax` exponentiates: `exp(-inf)` is 0, so masked positions get exactly zero weight, while the remaining (past and present) positions renormalize among themselves to still sum to 1. Passing `causal=True` tells `scaled_dot_product_attention` to apply this mask for us, and the printed weight matrix should be lower-triangular with a zeroed upper triangle.

In [ ]:
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
q = torch.randn(1, 4, 8)
k = torch.randn(1, 4, 8)
v = torch.randn(1, 4, 8)
out, w = scaled_dot_product_attention(q, k, v, causal=True)
print("causal weights (upper triangle is 0):")
print(w[0].round(decimals=2))

**What just happened.** This is proof, not promise. We take the keys and values of every token *after* the first and shove them by +5.0, a huge perturbation. Then we recompute attention and compare position 0's output before and after. It prints `True`: the first token's output did not move one bit, because under the causal mask token 0 never attended to anything past itself. The second check confirms the upper triangle of the weight matrix is exactly zero. The past genuinely cannot see the future, and we can demonstrate it rather than just assert it.

⚠️ **Common pitfalls**
- Masking with 0 instead of `-inf`. Setting masked scores to 0 still leaves them as `exp(0) = 1` after softmax, so the future leaks. The mask must be applied as `-inf` *before* the softmax.
- Off-by-one on the diagonal. `diagonal=1` keeps the main diagonal unmasked (a token may attend to itself) and masks strictly above it. Using `diagonal=0` would wrongly forbid self-attention.
- Applying the mask after softmax. Order matters: mask, then softmax, then renormalize. Masking afterward breaks the row-sums-to-1 property.

🏋️ **Try it yourself.** Change the perturbation target from `[:, 1:]` to `[:, 2:]` and check position 1's output instead of position 0. It should also be unchanged, because position 1 only sees tokens 0 and 1. Then flip `causal=False` and watch the leak appear: position 0's output now moves when you edit the future.

In [ ]:
k2 = k.clone()
v2 = v.clone()
k2[:, 1:] += 5.0
v2[:, 1:] += 5.0
out2, _ = scaled_dot_product_attention(q, k2, v2, causal=True)
print("position 0 unchanged by future edits:", torch.allclose(out[:, 0], out2[:, 0]))
upper = torch.triu(torch.ones(4, 4), diagonal=1).bool()
assert torch.all(w[0][upper] == 0)
assert torch.allclose(out[:, 0], out2[:, 0], atol=1e-6)